# 05 — Stage 3: Hardened Multi-Signal Attestation Check

**Goal.** Detect forged Stage 2a and Stage 2b models that pass naive public-fingerprint cosine checks by combining:
1. Public refusal-direction fingerprint cosine (`fingerprint_cosine`)
2. Refusal-circuit top-component Jaccard overlap (`circuit_overlap`)
3. Secret multilingual probe refusal rate (`probe_refusal_rate`)
4. SHA-256 cryptographic commitment verification (`verify_commitment`)

In [ ]:
!git clone https://github.com/mathcrypto/forged-fingerprints.git /content/forged-fingerprints 2>/dev/null || (cd /content/forged-fingerprints && git pull)
%cd /content/forged-fingerprints
!pip install -q -e . peft accelerate

In [ ]:
import os
from huggingface_hub import login, snapshot_download

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')
login(token=token) if token else login()

HF_ORG = 'team-doom'
snapshot_download(f'{HF_ORG}/forged-fingerprints-contrast-public', repo_type='dataset', local_dir='data/contrast_sets/public')
try:
    snapshot_download(f'{HF_ORG}/forged-fingerprints-probes-secret', repo_type='dataset', local_dir='data/contrast_sets/secret')
except Exception as e:
    print('Using held-out split B fallback if secret probes unavailable:', e)
snapshot_download(f'{HF_ORG}/forged-fingerprints-results', repo_type='dataset', local_dir='outputs')

## 1. Run Hardened Check Across Reference, Benign, and Forged Attack Models

In [ ]:
from pathlib import Path
import subprocess

# Check clean reference model
subprocess.run(['python', 'scripts/run_hardened_check.py', '--out', 'outputs/checks/base.json'], check=True)

# Check all trained adapters in zoo and attacks
for adapter_dir in sorted(list(Path('outputs/zoo').glob('*/adapter')) + list(Path('outputs/attacks').glob('*/adapter'))):
    run_name = adapter_dir.parent.name
    out_file = f'outputs/checks/{run_name}.json'
    print(f'=== Auditing {run_name} ===')
    subprocess.run([
        'python', 'scripts/run_hardened_check.py',
        '--adapter', str(adapter_dir),
        '--out', out_file
    ], check=True)

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

print(f"{'model':28s} {'fp_cos':>8s} {'circuit':>8s} {'probe_ref':>10s} {'flagged':>8s}  reasons")
print('-' * 90)
for p in sorted(Path('outputs/checks').glob('*.json')):
    r = json.loads(p.read_text())
    print(f"{p.stem:28s} {r['fingerprint_cosine']:8.4f} {r['circuit_overlap']:8.4f} {r['probe_refusal_rate']:10.3f} {str(r['flagged']):>8s}  {', '.join(r['reasons'])}")

HfApi().upload_folder(folder_path='outputs/checks', path_in_repo='checks', repo_id=f'{HF_ORG}/forged-fingerprints-results', repo_type='dataset')